In [1]:
import os


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from typing import Literal
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver

In [5]:
import sqlite3

In [6]:
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)

In [7]:
memory = SqliteSaver(conn)

In [8]:
from langchain_core.messages import SystemMessage
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [9]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


In [10]:
def research(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    system = SystemMessage(content="You are a helpful and friendly assistant that does research")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

In [11]:
builder = StateGraph(State)

In [12]:
       
builder.add_node("chat", research)

In [13]:
builder.add_edge(START, "chat")

In [14]:
builder.add_edge("chat", END)

In [15]:
router = builder.compile(checkpointer=memory)

In [16]:
config = {"configurable": {"thread_id": "user_123"}}

In [17]:
result1 = router.invoke(
    {"messages": [HumanMessage(content="list 5 top universities in Africa")]},
    config=config            # ← thread_id = "user_123"
)

In [18]:
print(result1["messages"][-1].content)

Here are five top universities in Africa, known for their academic excellence and research contributions:

1. **University of Cape Town (UCT)** - Located in South Africa, UCT is often ranked as the best university in Africa and is renowned for its research output and academic programs across various fields.

2. **University of the Witwatersrand (Wits)** - Also in South Africa, Wits is recognized for its strong emphasis on research and offers a range of undergraduate and postgraduate programs. It has a rich history and a vibrant campus culture.

3. **Stellenbosch University** - This South African university is known for its quality research, especially in the areas of agriculture, economics, and engineering. It has a strong international reputation and is one of the oldest universities in South Africa.

4. **American University in Cairo (AUC)** - AUC is a leading institution in Egypt that provides an American-style education in the heart of the Arab world. It is known for its liberal ar